Stuart Rossiter, C00284845

# Licensing
Recurrent Neural Network to predict Donald Trump's next tweet with the Trump Tweets dataset on Kaggle Copyright (C) 2025 Stuart Rossiter 
https://www.kaggle.com/datasets/austinreese/trump-tweets

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program. If not, see https://www.gnu.org/licenses/.

# Recurrent Neural Networks - Trump Tweet Prediction
Using a Recurrent Neural Network, I will be predicting Donald Trump's next big tweet.

# Preprocessing
First things first, we need to take in the data. We'll be using the tweet id, and its contents. The model will be trained with no URLs, but with hashtags and mentions. These will be named Authentic Tweets.

In [8]:
import pandas as pd

df = pd.read_csv("trumptweets.csv", usecols=["id", "content"])
df = df.dropna()
authentic_tweets = df.copy()

authentic_tweets['content'] = authentic_tweets['content'].str.replace(r'http\S+', '', regex=True).str.strip()

Next, we need to tokenize the words. The model doesn't understand words, so we assign them values instead.

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
auth_tokenizer = Tokenizer()
auth_tokenizer.fit_on_texts(authentic_tweets['content'])

We can't have the model just spitting random words in random orders, so we generate sequences to train the model on.

In [10]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

sequences = []
for tweet in authentic_tweets['content']:
    token_list = auth_tokenizer.texts_to_sequences([tweet])[0]
    for i in range(1, len(token_list)):
        sequences.append(token_list[:i+1])

sequences = pad_sequences(sequences)
X_auth = sequences[:, :-1]
y_auth = sequences[:, -1]
max_len_auth = sequences.shape[1]

With the data sequenced, it's time to create our model.

# Creating the Recurrent Neural Network Structure
It's time to create our model. This will be used to generate the tweets.

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

authModel = Sequential([Embedding(len(auth_tokenizer.word_index) + 1, 64, input_length=max_len_auth - 1),LSTM(128, return_sequences=True),Dropout(0.2),LSTM(64),Dense(len(auth_tokenizer.word_index) + 1, activation='softmax')])
authModel.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


# Training
Now that we've created the models, it's time to train. Let's train the model on Authentic Tweets.

In [7]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

callbacks = [EarlyStopping(patience=3, restore_best_weights=True),ModelCheckpoint('best_model_authentic.keras', save_best_only=True),]

history_authentic = authModel.fit(X_auth, y_auth,epochs=10,batch_size=128,validation_split=0.2,callbacks=callbacks)

Epoch 1/10
4751/4751 ━━━━━━━━━━━━━━━━━━━━ 645s 135ms/step - accuracy: 0.0675 - loss: 6.9796 - val_accuracy: 0.1069 - val_loss: 6.5958
Epoch 2/10
4751/4751 ━━━━━━━━━━━━━━━━━━━━ 661s 139ms/step - accuracy: 0.1150 - loss: 6.3064 - val_accuracy: 0.1265 - val_loss: 6.3862
Epoch 3/10
4751/4751 ━━━━━━━━━━━━━━━━━━━━ 665s 140ms/step - accuracy: 0.1334 - loss: 5.9822 - val_accuracy: 0.1364 - val_loss: 6.2906
Epoch 4/10
4751/4751 ━━━━━━━━━━━━━━━━━━━━ 655s 138ms/step - accuracy: 0.1471 - loss: 5.7412 - val_accuracy: 0.1457 - val_loss: 6.2298
Epoch 5/10
4751/4751 ━━━━━━━━━━━━━━━━━━━━ 657s 138ms/step - accuracy: 0.1568 - loss: 5.5543 - val_accuracy: 0.1483 - val_loss: 6.2275
Epoch 6/10
4751/4751 ━━━━━━━━━━━━━━━━━━━━ 664s 140ms/step - accuracy: 0.1643 - loss: 5.4065 - val_accuracy: 0.1527 - val_loss: 6.2160
Epoch 7/10
4751/4751 ━━━━━━━━━━━━━━━━━━━━ 668s 141ms/step - accuracy: 0.1710 - loss: 5.2844 - val_accuracy: 0.1553 - val_loss: 6.2362
Epoch 8/10
4751/4751 ━━━━━━━━━━━━━━━━━━━━ 682s 141ms/step - ac

# Predictions
Let's make a function for predicting the continuation of a tweet. This way, we can reuse it later to predict all sorts of things without repeating the code.

In [34]:
import numpy as np

def generate_tweet(seed, tokenizer, model, max_len, num_words=20, temperature=1):
    result = seed
    for _ in range(num_words):
        token_list = tokenizer.texts_to_sequences([result])[0]
        token_list = pad_sequences([token_list], maxlen=max_len - 1)
        
        predictions = model.predict(token_list, verbose=0)[0]
        
        # temperature scales creativity — lower = safer, higher = weirder
        predictions = np.log(predictions + 1e-7) / temperature
        predictions = np.exp(predictions) / np.sum(np.exp(predictions))
        
        predicted_index = np.random.choice(len(predictions), p=predictions)
        
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                result += " " + word
                break
    
    return result

Let's test it out. We'll provide the seed of "MAKE AMERICA" and see what happens. Ideally, GREAT AGAIN should follow after.

In [62]:
print(generate_tweet("MAKE AMERICA", auth_tokenizer, authModel, max_len_auth, num_words=10))

MAKE AMERICA donaldjtrumpjr bad’” kavemanrj mazk5bdzud heiressarts pat everybody's sugarcoat asad shipments


This was an unfortunate but not unforeseen issue. The inclusion of hashtags and mentions (@s) skews the data by including potential gibberish tags and usernames. Some words sneak through, but the model is corrupted by these values. Let's try a few more, messing with the temperature and max words.

In [78]:
print(generate_tweet("MAKE AMERICA", auth_tokenizer, authModel, max_len_auth, num_words=3, temperature=0.2))
print(generate_tweet("MAKE AMERICA", auth_tokenizer, authModel, max_len_auth, num_words=3, temperature=0.5))
print(generate_tweet("MAKE AMERICA", auth_tokenizer, authModel, max_len_auth, num_words=5, temperature=1.5))
print(generate_tweet("MAKE AMERICA", auth_tokenizer, authModel, max_len_auth, num_words=1, temperature=2))
print(generate_tweet("MAKE AMERICA", auth_tokenizer, authModel, max_len_auth, num_words=15, temperature=10))

MAKE AMERICA tvashleigh elvisknievil theresanov68
MAKE AMERICA tammieharrison shutdown—an westjournalism
MAKE AMERICA rydev22 marginalize kcbonfeibv dilemma ladyzales
MAKE AMERICA guns
MAKE AMERICA references biggangvh1 twan permissible atitude writeonright dominance ‘trump restoration v8vlxdd0fo back2reason georges chrispalmer bretmichaels point's


Unfortunately, it still seems to predict gibberish. The most clear result is MAKE AMERICA guns. One could argue its relevancy, but the model is not accurate enough.

# Conclusion
Unfortunately, the model is subpar, at best. More Epochs are needed to train it, but that would take hours, if not days. In future, I would omit the hashtags and mentions, to hopefully get a clearer result.